In [1]:
import random
import math

# ==========================================
# 0. FUNCIONES BASE (Optimizadas)
# ==========================================
def distancia(p1, p2):
    suma = sum((a - b) ** 2 for a, b in zip(p1, p2))
    return math.sqrt(suma)

def centroide(puntos):
    n = len(puntos)
    if n == 0:
        return [0] * len(puntos[0])
    dim = len(puntos[0])
    suma = [0] * dim
    for p in puntos:
        for i in range(dim):
            suma[i] += p[i]
    return [s / n for s in suma]

def sse(clusters, centroids):
    total = 0
    for i, cluster in enumerate(clusters):
        for point in cluster:
            # SSE = Suma de la distancia AL CUADRADO
            total += distancia(point, centroids[i]) ** 2
    return total

def kmeans(data, k, max_iters=100):
    # Asegurarnos de no pedir más clusters que datos
    k = min(k, len(data))
    centroids = random.sample(data, k)

    for _ in range(max_iters):
        clusters = [[] for _ in range(k)]

        # Asignar puntos al centroide más cercano
        for point in data:
            distancias = [distancia(point, c) for c in centroids]
            indice_min = distancias.index(min(distancias))
            clusters[indice_min].append(point)
            
        # Recalcular centroides
        nuevos_centroids = []
        for i, cluster in enumerate(clusters):
            # Prevención de cluster vacío: mantiene el centroide anterior
            if len(cluster) == 0:
                nuevos_centroids.append(centroids[i])
            else:
                nuevos_centroids.append(centroide(cluster))

        if nuevos_centroids == centroids:
            break
        centroids = nuevos_centroids

    return clusters, centroids

# DATOS
data = [
    [1, 1],       # A
    [1.5, 2],     # B
    [3, 4],       # C
    [5.5, 7],     # D
    [3.5, 5],     # E
    [4.5, 5],     # F
    [3.5, 4.5],   # G
]

# ==========================================
# 1 Y 2. ITERAR Y CALCULAR SSE PARA CADA k
# ==========================================
# Solo iteramos hasta el tamaño de la muestra o 10 (lo que sea menor)
k_maximo = min(10, len(data)) 
sse_valores = []

print("Calculando SSE para diferentes valores de K...")
for k in range(1, k_maximo + 1):
    mejor_sse = float('inf')
    # Ejecutamos K-Means múltiples veces para cada k para evitar mínimos locales por inicialización aleatoria
    for _ in range(15): 
        clusters_tmp, centroids_tmp = kmeans(data, k)
        error_actual = sse(clusters_tmp, centroids_tmp)
        if error_actual < mejor_sse:
            mejor_sse = error_actual
    
    sse_valores.append(mejor_sse)
    print(f" k={k} \t SSE={mejor_sse:.4f}")

# ==========================================
# 3. DETECTAR EL CODO AUTOMÁTICAMENTE
# ==========================================
# Método: Mayor distancia a la recta entre el primer (k=1) y último (k=k_maximo) punto
x1, y1 = 1, sse_valores[0]
x2, y2 = k_maximo, sse_valores[-1]

distancia_maxima = -1
k_optimo = 1

for i in range(k_maximo):
    x0 = i + 1
    y0 = sse_valores[i]
    
    # Fórmula geométrica de distancia de un punto (x0,y0) a la línea P1-P2
    numerador = abs((y2 - y1)*x0 - (x2 - x1)*y0 + x2*y1 - y2*x1)
    denominador = math.sqrt((y2 - y1)**2 + (x2 - x1)**2)
    distancia_a_recta = numerador / denominador
    
    if distancia_a_recta > distancia_maxima:
        distancia_maxima = distancia_a_recta
        k_optimo = x0

print(f"\n=> ¡Codo detectado automáticamente en k = {k_optimo}!")

# ==========================================
# 4. IMPRIMIR RESULTADOS DEL K ÓPTIMO
# ==========================================
print(f"\n--- Ejecución final con el k óptimo ({k_optimo}) ---")

# Buscamos el mejor particionamiento para el k óptimo
mejor_sse_final = float('inf')
mejores_clusters = None
mejores_centroides = None

for _ in range(20):
    c_final, ct_final = kmeans(data, k_optimo)
    s_final = sse(c_final, ct_final)
    if s_final < mejor_sse_final:
        mejor_sse_final = s_final
        mejores_clusters = c_final
        mejores_centroides = ct_final

for i, cluster in enumerate(mejores_clusters):
    print(f"\nCluster {i + 1}:")
    for punto in cluster:
        print("  ", punto)
    print(" → Centroide:", [round(c, 2) for c in mejores_centroides[i]])

Calculando SSE para diferentes valores de K...
 k=1 	 SSE=39.1429
 k=2 	 SSE=9.8250
 k=3 	 SSE=2.5000
 k=4 	 SSE=1.2917
 k=5 	 SSE=0.6667
 k=6 	 SSE=0.1250
 k=7 	 SSE=0.0000

=> ¡Codo detectado automáticamente en k = 3!

--- Ejecución final con el k óptimo (3) ---

Cluster 1:
   [1, 1]
   [1.5, 2]
 → Centroide: [1.25, 1.5]

Cluster 2:
   [3, 4]
   [3.5, 5]
   [4.5, 5]
   [3.5, 4.5]
 → Centroide: [3.62, 4.62]

Cluster 3:
   [5.5, 7]
 → Centroide: [5.5, 7.0]
